# TP3 - Couche Bronze dans PostgreSQL

Ingestion de telemetry.csv et releves_incidents.csv avec flag de coherence sur les doublons horaires.

## 0. Connexion

In [ ]:
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
import pandas as pd

db_url = URL.create(
    drivername='postgresql+psycopg2',
    username='indusense_user',
    password='ThEP@ssW0rd',
    host='localhost',
    port=5432,
    database='indusense_db',
)
engine = create_engine(db_url)
with engine.connect() as conn:
    print(conn.execute(text('SELECT version()')).scalar())


## 1. Chargement des CSV

In [ ]:
df_tel = pd.read_csv('telemetry.csv', dtype=str)
df_inc = pd.read_csv('releves_incidents.csv', dtype=str, encoding='utf-8')
print(f'Telemetry : {df_tel.shape}')
print(f'Incidents : {df_inc.shape}')
df_tel.head()


## 2. Validation minimale Bronze

In [ ]:
EXPECTED_TEL = {'machine_id','timestamp','temperature_c','pressure_bar',
                'voltage_mean_v','rotation_mean_rpm','pieces_produced'}
EXPECTED_INC = {'incident_id','date','time','operator_name','machine_id',
                'severity','operator_badge','comment','shift',
                'type_surchauffe','type_baisse_pression','type_vibration',
                'type_bruit_mecanique','type_surconsommation','type_blocage_mecanique',
                'type_alarme_capteur','type_arret_urgence','type_defaut_qualite'}
assert not EXPECTED_TEL - set(df_tel.columns)
assert not EXPECTED_INC - set(df_inc.columns)
print('Validation Bronze OK')


## 3. Flag de coherence - un seul releve par heure par machine

Regle : exactement un releve par (machine_id, timestamp).  
- `parse_ok` = True par defaut  
- `parse_ok` = False + `parse_ok_reason` si doublon detecte


In [ ]:
counts = df_tel.groupby(['machine_id', 'timestamp'])['machine_id'].transform('count')

df_tel['parse_ok']        = counts == 1
df_tel['parse_ok_reason'] = ''

mask_dup = counts > 1
df_tel.loc[mask_dup, 'parse_ok_reason'] = (
    'Doublon : ' + counts[mask_dup].astype(str)
    + ' releves pour ' + df_tel.loc[mask_dup, 'machine_id']
    + ' a ' + df_tel.loc[mask_dup, 'timestamp']
)

print(f'parse_ok=True  : {(~mask_dup).sum():,} lignes')
print(f'parse_ok=False : {mask_dup.sum():,} lignes ({mask_dup.sum()/len(df_tel)*100:.1f}%)')
print()
display(df_tel[mask_dup][['machine_id','timestamp','parse_ok','parse_ok_reason']].head(4))


## 3b. Flag de coherence — incidents

**Regle 1** : `severity` doit etre compris entre 1 et 5 inclus.  
**Regle 2** : les 9 colonnes `type_*` doivent valoir 0 ou 1.


In [ ]:
TYPE_COLS = [
    'type_surchauffe','type_baisse_pression','type_vibration',
    'type_bruit_mecanique','type_surconsommation','type_blocage_mecanique',
    'type_alarme_capteur','type_arret_urgence','type_defaut_qualite'
]

df_inc['parse_ok']        = True
df_inc['parse_ok_reason'] = ''

# Regle 1 : severity hors [1-5]
sev = pd.to_numeric(df_inc['severity'], errors='coerce')
mask_sev = sev.isna() | (sev < 1) | (sev > 5)
df_inc.loc[mask_sev, 'parse_ok'] = False
df_inc.loc[mask_sev, 'parse_ok_reason'] += (
    'Regle1 : severity=' + df_inc.loc[mask_sev, 'severity'].astype(str) + ' hors [1-5]. '
)

# Regle 2 : type_* hors {0, 1}
for col in TYPE_COLS:
    v = pd.to_numeric(df_inc[col], errors='coerce')
    mask_col = v.isna() | ~v.isin([0, 1])
    df_inc.loc[mask_col, 'parse_ok'] = False
    df_inc.loc[mask_col, 'parse_ok_reason'] += (
        f'Regle2 : {col}=' + df_inc.loc[mask_col, col].astype(str) + ' hors {0,1}. '
    )

n_nok = (~df_inc['parse_ok']).sum()
print(f'parse_ok=True  : {df_inc["parse_ok"].sum():,} lignes')
print(f'parse_ok=False : {n_nok:,} lignes')
if n_nok:
    display(df_inc[~df_inc['parse_ok']][['incident_id','severity','parse_ok','parse_ok_reason']].head())


## 4. Typage minimal

In [ ]:
df_tel_typed = df_tel.copy()
for col in ['temperature_c','pressure_bar','voltage_mean_v','rotation_mean_rpm']:
    df_tel_typed[col] = pd.to_numeric(df_tel_typed[col], errors='coerce')
df_tel_typed['pieces_produced'] = pd.to_numeric(df_tel_typed['pieces_produced'], errors='coerce').astype('Int64')

df_inc_typed = df_inc.copy()
for col in ['severity','type_surchauffe','type_baisse_pression','type_vibration',
            'type_bruit_mecanique','type_surconsommation','type_blocage_mecanique',
            'type_alarme_capteur','type_arret_urgence','type_defaut_qualite']:
    df_inc_typed[col] = pd.to_numeric(df_inc_typed[col], errors='coerce').astype('Int64')

print('Typage effectue')


## 5. Ingestion dans PostgreSQL

Alembic gere le schema. On vide les tables (TRUNCATE) puis on insere.


In [ ]:
with engine.begin() as conn:
    conn.execute(text('TRUNCATE TABLE bronze_telemetry RESTART IDENTITY'))
    conn.execute(text('TRUNCATE TABLE bronze_incidents  RESTART IDENTITY'))

df_tel_typed.to_sql('bronze_telemetry', engine, if_exists='append',
                    index=False, method='multi', chunksize=1000)
df_inc_typed.to_sql('bronze_incidents', engine, if_exists='append',
                    index=False, method='multi', chunksize=1000)

print(f'bronze_telemetry : {len(df_tel_typed):,} lignes')
print(f'bronze_incidents  : {len(df_inc_typed):,} lignes')


## 6. Verification

In [ ]:
with engine.connect() as conn:
    n_tel     = conn.execute(text('SELECT COUNT(*) FROM bronze_telemetry')).scalar()
    n_tel_ok  = conn.execute(text('SELECT COUNT(*) FROM bronze_telemetry WHERE parse_ok = true')).scalar()
    n_tel_nok = conn.execute(text('SELECT COUNT(*) FROM bronze_telemetry WHERE parse_ok = false')).scalar()
    n_inc     = conn.execute(text('SELECT COUNT(*) FROM bronze_incidents')).scalar()
    n_inc_ok  = conn.execute(text('SELECT COUNT(*) FROM bronze_incidents WHERE parse_ok = true')).scalar()
    n_inc_nok = conn.execute(text('SELECT COUNT(*) FROM bronze_incidents WHERE parse_ok = false')).scalar()

print(f'bronze_telemetry : {n_tel:,} lignes  |  OK: {n_tel_ok:,}  |  Doublons: {n_tel_nok:,}')
print(f'bronze_incidents  : {n_inc:,} lignes  |  OK: {n_inc_ok:,}  |  KO: {n_inc_nok:,}')
print()
print('-- Telemetry : exemples parse_ok=False --')
display(pd.read_sql(
    'SELECT machine_id, timestamp, parse_ok, parse_ok_reason FROM bronze_telemetry WHERE parse_ok = false LIMIT 3',
    engine
))
print('-- Incidents : exemples parse_ok=False --')
display(pd.read_sql(
    'SELECT incident_id, severity, parse_ok, parse_ok_reason FROM bronze_incidents WHERE parse_ok = false LIMIT 3',
    engine
))
